

### **Finetuning Llama-3.2-1B**



In [1]:
!pip install --upgrade huggingface_hub
!pip install transformers
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.6/515.6 kB 31.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.1 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.1.4 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 36.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.1.4
    Uninstalling huggingface_hub-1.1.4:
      Successfully uninstalled huggingface_hub-1.1.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 35.6 MB/s eta 0:00:00


#### Importar librerías requeridas

In [2]:
import transformers
import torch
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)
import numpy as np
import time
import matplotlib.pyplot as plt
from datasets import load_dataset
from datasets import DatasetDict

import numpy as np
import pandas as pd
from tqdm import tqdm
import os

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")

seed = 136
torch.manual_seed(seed)
np.random.seed(seed)

MODEL_ID = "meta-llama/Llama-3.2-1B"
BLOCK_SIZE = 256

BATCH_SIZE = 4
ACCUM_GRAD_STEPS = 4


login(token="insert_your_huggingface_token_here")



GPU: NVIDIA A100-SXM4-80GB


#### Carga del dataset RACE

Se carga el dataset y se hace un filtro para considerar únicamente artículos con 800 tokens o menos.

In [3]:
raw = load_dataset("ehovy/race", "all")   # train, validation, test

def filter_short(example):
    return len(example["article"]) < 800

def filter_and_keep(example):
    if filter_short(example):
        return example
    return None

filtered = raw.filter(filter_short)

print(filtered)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/2.08M [00:00<?, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/37.4M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/2.05M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/4887 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4934 [00:00<?, ? examples/s]

Filter:   0%|          | 0/87866 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4887 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['example_id', 'article', 'answer', 'question', 'options'],
        num_rows: 480
    })
    train: Dataset({
        features: ['example_id', 'article', 'answer', 'question', 'options'],
        num_rows: 8930
    })
    validation: Dataset({
        features: ['example_id', 'article', 'answer', 'question', 'options'],
        num_rows: 514
    })
})


#### Construcción del prompt

Se carga el dataset y se hace un filtro para considerar únicamente artículos con 800 tokens o menos. La función `build_mcq_prompt` onstruye un prompt tipo instrucción para que el modelo responda con una sola letra (A, B, C o D).

In [4]:
def build_prompt(example):
    article = example["article"]
    question = example["question"]
    options = example["options"]
    answer = example["answer"]

    prompt = f"""### Article:
      {article}

      ### Question:
      {question}

      ### Options:
      A) {options[0]}
      B) {options[1]}
      C) {options[2]}
      D) {options[3]}

      ### Answer (one letter):"""
    return {"prompt": prompt, "answer": answer}


In [5]:
filtered = filtered.map(build_prompt)
print(filtered)

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

Map:   0%|          | 0/8930 [00:00<?, ? examples/s]

Map:   0%|          | 0/514 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['example_id', 'article', 'answer', 'question', 'options', 'prompt'],
        num_rows: 480
    })
    train: Dataset({
        features: ['example_id', 'article', 'answer', 'question', 'options', 'prompt'],
        num_rows: 8930
    })
    validation: Dataset({
        features: ['example_id', 'article', 'answer', 'question', 'options', 'prompt'],
        num_rows: 514
    })
})


#### Evaluador por probabilidad (modelo base)

La función `tokenize_function` prepara datos para el entrenamiento: primero concatena cada prompt con su respuesta y tokeniza el texto completo con un máximo de 1024 tokens; luego construye manualmente los labels para que el modelo aprenda únicamente a generar la respuesta, asignando -100 a todos los tokens del prompt y del padding (para que el loss los ignore) y manteniendo solo los input_ids de la respuesta como objetivos de entrenamiento.

In [6]:
model_id = "meta-llama/Llama-3.2-1B"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    # Concatenar prompt + respuesta correcta
    full_texts = [p + " " + a for p, a in zip(examples["prompt"], examples["answer"])]
    tokenized = tokenizer(
        full_texts,
        truncation=True,
        max_length=1024,
        padding="max_length",
        return_tensors=None,
    )

    # Crear labels: -100 para el prompt, tokens reales para la respuesta
    labels = []
    for i, (prompt, answer) in enumerate(zip(examples["prompt"], examples["answer"])):
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        answer_ids = tokenizer(" " + answer, add_special_tokens=False)["input_ids"]
        label = [-100] * len(prompt_ids) + answer_ids + [-100] * (1024 - len(prompt_ids) - len(answer_ids))
        labels.append(label[:1024])
    tokenized["labels"] = labels
    return tokenized

tokenized = filtered.map(tokenize_function, batched=True, remove_columns=filtered["train"].column_names)


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

Map:   0%|          | 0/8930 [00:00<?, ? examples/s]

Map:   0%|          | 0/514 [00:00<?, ? examples/s]

#### Configuración LoRA

Se utilizará LoRA para realizar el fine-tuning, LoRA permite añadir matrices de bajo rango que suman a las proyecciones originales del modelo, permitiendo adaptarlo entrenando muchos menos parámetros ya que sólo los parámetros de `lora_config` son entrenados, mientras los demás permanecen congelados.

Primero se define una configuración de cuantización en 4 bits (NF4, cómputo en float16 y double quantization), luego se carga el modelo causal usando esa configuración y asignándolo automáticamente a los dispositivos disponibles. Después, se especifica la configuración LoRA con rango 64, α=16, dropout 0.05 y apuntando a los módulos clave de atención (q_proj, k_proj, v_proj, o_proj). Finalmente, se envuelve el modelo con get_peft_model para habilitar el entrenamiento eficiente y se imprime cuántos parámetros resultan entrenables.

In [7]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
)

lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

trainable params: 13,631,488 || all params: 1,249,445,888 || trainable%: 1.0910


#### Entrenamiento del modelo Fine-tuning con LoRA

Se crea un Trainer pasando el modelo, los argumentos, los conjuntos de entrenamiento y validación, el collator y el tokenizador definidos anteriormente.

Se utiliza `weight_decay=0.01` para evitar el sobreajuste añadiendo una regularización ligera. Se usa un `warmup=0.1` para ayudar al optimizador a estabilizadrse y evitar saltos brusocs en el entrenamiento.

In [8]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./llama32_1b_race_lora",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE*2,
    gradient_accumulation_steps=ACCUM_GRAD_STEPS,
    learning_rate=2e-4,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    eval_steps=200,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    warmup_ratio=0.1,
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)
trainer.train()





/tmp/ipython-input-833655563.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Epoch,Training Loss,Validation Loss
1,1.672700,1.765015
2,1.412200,1.784458
3,1.272300,1.855750


TrainOutput(global_step=1677, training_loss=1.5219993124991993, metrics={'train_runtime': 1593.6607, 'train_samples_per_second': 16.81, 'train_steps_per_second': 1.052, 'total_flos': 1.62421382578176e+17, 'train_loss': 1.5219993124991993, 'epoch': 3.0})

#### Predicción de respuestas del modelo Fine-tuning

La función `build_inference_prompt` recibe un artículo, una pregunta y cuatro opciones múltiples, y crea un prompt ("Article", "Question", "Options") que facilitan al modelo entender el contexto y producir solo una letra al final.

Luego, `predict_answer_generation` tokeniza ese prompt, lo envía al modelo en modo de inferencia sin gradientes (torch.no_grad()), y usa model.generate sin muestreo (do_sample=False) para obtener una predicción.

El modelo genera solo unos pocos tokens (por defecto 2) justo después de la sección "### Answer (one letter):". Finalmente, la función decodifica la salida, extrae el texto posterior a esa etiqueta y toma únicamente la primera letra, asegurando que la respuesta final sea una opción válida (A, B, C o D).

In [9]:
def build_inference_prompt(article, question, options):
    prompt = f"""### Article:
    {article}

    ### Question:
    {question}

    ### Options:
    A) {options[0]}
    B) {options[1]}
    C) {options[2]}
    D) {options[3]}

    ### Answer (one letter):"""
    return prompt

def predict_answer_generation(model, prompt, max_new_tokens=2):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            inputs["input_ids"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    answer_part = generated.split("### Answer (one letter):")[-1].strip()

    # Extraer solo la primera letra
    predicted_letter = answer_part[0].upper() if answer_part else "?"

    return predicted_letter

#### Evaluador de probabilidades

La función `predict_answer_probability` calcula cuál de las cuatro opciones (A, B, C o D) es más probable según el modelo, evaluando explícitamente la probabilidad de cada letra en el contexto del prompt de inferencia.

Primero construye el prompt usando `build_inference_prompt` y lo tokeniza, luego define la lista de opciones posibles. Para cada letra, crea la secuencia completa prompt + letra, tokeniza solo la letra para conocer sus IDs, concatena ambos y pasa la secuencia completa por el modelo sin gradientes.

Después obtiene los logits únicamente de los tokens correspondientes a esa letra y les aplica log_softmax para convertirlos en log-probabilidades. Extrae la log-probabilidad real de cada token de la letra y calcula su media, actuando como una puntuación de qué tan probable es que el modelo genere esa opción en ese contexto.

Finalmente, compara las cuatro puntuaciones y devuelve la letra con el mayor valor, que corresponde a la opción más probable según el modelo.

In [10]:
import numpy as np

def predict_answer_probability(model, article, question, options, tokenizer):
    context = build_inference_prompt(article, question, options)
    context_ids = tokenizer(context, return_tensors="pt").input_ids.to(model.device)

    logprobs = []
    letters = ["A", "B", "C", "D"]

    for letter in letters:
        opt_text = " " + letter
        opt_ids = tokenizer(opt_text, add_special_tokens=False, return_tensors="pt").input_ids.to(model.device)

        input_ids = torch.cat([context_ids, opt_ids], dim=1)

        with torch.no_grad():
            outputs = model(input_ids)

            logits = outputs.logits[0, -len(opt_ids):, :]

        log_prob = torch.log_softmax(logits, dim=-1)

        token_logprob = log_prob.gather(1, opt_ids.squeeze(0).unsqueeze(-1)).squeeze(-1)
        avg_logprob = token_logprob.mean().item()

        logprobs.append(avg_logprob)

    return letters[np.argmax(logprobs)]


#### Evaluación del modelo base

Se evalúa el modelo original sin LoRA, primero se carga el modelo base en modo inferencia con FP16 y asignación automática de dispositivos. Luego recorre cada ejemplo del split test, extrayendo el artículo, la pregunta, las opciones y la respuesta correcta.

Para cada caso construye el prompt de inferencia y obtiene dos predicciones: (1) generación directa, donde el modelo completa el texto y entrega una letra; y (2) probabilidad condicional P(s|c), donde se calcula explícitamente la probabilidad del modelo para cada opción A–D y se elige la más probable.

 Después compara cada predicción con la respuesta real y acumula aciertos para ambos métodos. Finalmente imprime la exactitud (accuracy) de cada enfoque sobre el total del conjunto de prueba, permitiendo comparar qué método de inferencia funciona mejor para el modelo.

In [11]:

#modelo original
model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True
)

test = filtered["test"]

correct_gen = 0
correct_prob = 0
total = len(test)

for example in tqdm(test):
    article = example["article"]
    question = example["question"]
    options = example["options"]
    true_answer = example["answer"]

    prompt = build_inference_prompt(article, question, options)

    # Método 1: Generación
    pred_gen = predict_answer_generation(model, prompt)

    # Método 2: P(s|c)
    pred_prob = predict_answer_probability(model, article, question, options, tokenizer)

    if pred_gen == true_answer:
        correct_gen += 1
    if pred_prob == true_answer:
        correct_prob += 1

print(f"Accuracy (generación): {correct_gen/total:.1%}")
print(f"Accuracy (P(s|c)): {correct_prob/total:.1%}")


`torch_dtype` is deprecated! Use `dtype` instead!
  0%|          | 0/480 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
100%|██████████| 480/480 [00:58<00:00,  8.21it/s]

Accuracy (generación): 30.2%
Accuracy (P(s|c)): 24.6%


#### Evaluación Modelo Fine tuning

Se realiza la evaluación al modelo con Fine tuning de la misma forma que se hizo con el modelo base

In [12]:
#modelo con ajuste fino
model = trainer.model
model.eval()
test = filtered["test"]

correct_gen = 0
correct_prob = 0
total = len(test)

for example in tqdm(test):
    article = example["article"]
    question = example["question"]
    options = example["options"]
    true_answer = example["answer"]

    prompt = build_inference_prompt(article, question, options)

    # Método 1: Generación
    pred_gen = predict_answer_generation(model, prompt)

    # Método 2: P(s|c)
    pred_prob = predict_answer_probability(model, article, question, options, tokenizer)

    if pred_gen == true_answer:
        correct_gen += 1
    if pred_prob == true_answer:
        correct_prob += 1

print(f"Accuracy (generación): {correct_gen/total:.1%}")
print(f"Accuracy (P(s|c)): {correct_prob/total:.1%}")

100%|██████████| 480/480 [03:05<00:00,  2.59it/s]

Accuracy (generación): 64.6%
Accuracy (P(s|c)): 28.1%


#### Guardado de checkpoint final del modelo

In [13]:
model.save_pretrained("./final_lora_checkpoint")
tokenizer.save_pretrained("./final_lora_checkpoint")

from peft import PeftModel
merged = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(model_id, device_map="auto"), "./final_lora_checkpoint")
merged = merged.merge_and_unload()
merged.save_pretrained("./final_merged_model")

In [18]:
lora_model_final = AutoModelForCausalLM.from_pretrained("./final_merged_model", torch_dtype=torch.float16)
torch.save(lora_model_final.state_dict(), "llama32_race_lora_merged.pt")

#### Análisis Cualitativo

La función `get_option_probabilities` calcula la probabilidad de cada respuesta (A–D) generando el prompt, concatenando cada posible letra y obteniendo los logits del modelo; luego transforma las log-probabilidades en probabilidades normalizadas para cada opción. La función `build_mcq_prompt_system` construye un prompt estilo sistema para que el modelo responda con una sola letra.

Finalmente, `evaluate_qualitative_comparison` recorre varios ejemplos del set de prueba, obtiene las probabilidades del modelo base y del modelo ajustado, organiza los resultados y los imprime en un formato legible, mostrando el prompt, las opciones, las probabilidades asignadas por ambos modelos y la respuesta correcta, lo que permite observar cómo cambia el comportamiento del modelo después del fine-tuning.

In [22]:
def get_option_probabilities(model, article, question, options, tokenizer):

    context = build_inference_prompt(article, question, options)
    context_ids = tokenizer(context, return_tensors="pt").input_ids.to(model.device)

    probabilities = {}
    letters = ["A", "B", "C", "D"]
    logprobs = []

    for letter in letters:
        opt_text = " " + letter
        opt_ids = tokenizer(opt_text, add_special_tokens=False, return_tensors="pt").input_ids.to(model.device)

        input_ids = torch.cat([context_ids, opt_ids], dim=1)

        with torch.no_grad():
            outputs = model(input_ids)
            logits = outputs.logits[0, -len(opt_ids):, :]

        log_prob = torch.log_softmax(logits, dim=-1)
        token_logprob = log_prob.gather(1, opt_ids.squeeze(0).unsqueeze(-1)).squeeze(-1)
        avg_logprob = token_logprob.mean().item()

        logprobs.append(avg_logprob)


    logprobs = np.array(logprobs)
    probs = np.exp(logprobs - np.max(logprobs))
    probs = probs / probs.sum()

    for i, letter in enumerate(letters):
        probabilities[letter] = probs[i]

    return probabilities


def build_mcq_prompt_system(article, question, options):

    prompt = f"""You are an AI system that answers multiple-choice questions based on a passage.

              Passage:
              {article}

              Question: {question}

              Options:
              A) {options[0]}
              B) {options[1]}
              C) {options[2]}
              D) {options[3]}

              Answer with the letter of the correct option (A, B, C, or D) only."""

    return prompt


def evaluate_qualitative_comparison(base_model, finetuned_model, tokenizer, test_dataset, num_examples=5):


    base_model.eval()
    finetuned_model.eval()

    results = []

    print(f"Evaluando {num_examples} ejemplos...\n")

    for idx in tqdm(range(min(num_examples, len(test_dataset)))):
        example = test_dataset[idx]

        article = example["article"]
        question = example["question"]
        options = example["options"]
        true_answer = example["answer"]

        # Construir el prompt en el formato del documento
        full_prompt = build_mcq_prompt_system(article, question, options)

        # Obtener probabilidades del modelo base
        base_probs = get_option_probabilities(base_model, article, question, options, tokenizer)

        # Obtener probabilidades del modelo fine-tuned
        ft_probs = get_option_probabilities(finetuned_model, article, question, options, tokenizer)


        result = {
            'index': idx + 1,
            'prompt': full_prompt,
            'options': options,
            'base_probs': base_probs,
            'ft_probs': ft_probs,
            'true_answer': true_answer
        }
        results.append(result)

    print("\n" + "=" * 56)
    print("COMPARACIÓN CUALITATIVA".center(56))
    print("=" * 56 + "\n")

    for result in results:
        print("\n" + "-" * 10 + f" EJEMPLO {result['index']} " + "-" * 10 + "\n")

        print("PREGUNTA:")
        print(result['prompt'] + "\n")

        print("OPCIONES:")
        for j, opt in enumerate(result['options']):
            letter = chr(65 + j)  # A, B, C, D
            print(f"  {letter}) {opt}")
        print()

        print("Probabilidades — Modelo Base:")
        for letter in ['A', 'B', 'C', 'D']:
            print(f"  {letter}: {result['base_probs'][letter]:.4f}")
        print()

        print("Probabilidades — Modelo LoRA:")
        for letter in ['A', 'B', 'C', 'D']:
            print(f"  {letter}: {result['ft_probs'][letter]:.4f}")
        print()

        print(f"RESPUESTA CORRECTA: {result['true_answer']}")

    return results




base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

finetuned_model = trainer.model


results = evaluate_qualitative_comparison(
    base_model=base_model,
    finetuned_model=finetuned_model,
    tokenizer=tokenizer,
    test_dataset=filtered["test"],
    num_examples=3,
)


Evaluando 3 ejemplos...



100%|██████████| 3/3 [00:01<00:00,  2.79it/s]


                COMPARACIÓN CUALITATIVA                 


---------- EJEMPLO 1 ----------

PREGUNTA:
You are an AI system that answers multiple-choice questions based on a passage.

Passage:
A new law helps people with disabilities. The law says that people with disabilities must be able to get into and out of all public buildings. It also says that business must offer special services to people who have special needs. Companies can not refuse to hire disabled workers. Many businesses may have to change their buildings and services.
--Ramps must be built so people can get into buildings.
--Movie theatres must have space for people in wheelchairs and seats for their friends to sit near them.
--Elevators must have floor number in  _ .
This law will help millions of people. One woman who has been in a wheelchair for many years said, "It is like a dream."

Question: According to the passage we can see that_.

Options:
A) it will be difficult for the normal persons to get into the public 

#### Conclusiones


Los resultadomuestran que el fine-tuning con LoRA aplicado al modelo Llama-3.2-1B, entrenado mediante prompts de RACE y una estrategia de enmascaramiento de labels que fuerza al modelo a aprender únicamente la respuesta final, produce una mejora sustancial en su capacidad de generación directa de respuestas, elevando la accuracy del 30.2% al 64.6%.

En conjunto, los resultados muestran que el modelo fine-tuning es altamente efectivo para inferencia por generación, mientras que la aproximación basada en probabilidad requiere ajustes adicionales o un enfoque de entrenamiento distinto para ser fiable.

Los resultados de la comparación cualitativa muestran que, aunque el modelo LoRA cambia de manera drástica las distribuciones de probabilidad y muestra muchísima más “confianza” en una sola opción, esta confianza no necesariamente corresponde a respuestas correctas. En todos los ejemplos, el modelo base produce distribuciones más balanceadas y cercanas al razonamiento esperado, aunque tampoco acierta consistentemente, mientras que el modelo fine-tuned concentra de forma extrema la probabilidad en una opción incorrecta en los tres casos evaluados, incluso cuando la respuesta correcta es clara en el contexto.

 Esto indica que el fine-tuning, al enfocarse solo en generar la letra final como objetivo, no optimiza adecuadamente la probabilidad token-por-token para clasificación, y de hecho puede inducir un sobreajuste que deteriora la calibración del modelo al evaluar P(s|c).
